In [1]:
import numpy as np
import pandas as pd

# ============================================================
# CONFUSION MATRICES (rows=Actual, cols=Predicted)
# Order: Normal(0), High Vol(1), Crash(3)
# ============================================================

cms = {
    'XGBoost': np.array([
        [163431, 16579,   276],
        [  4564, 53329,  2301],
        [    10,    49,   708],
    ]),
    'Random Forest': np.array([
        [159138, 20905,   243],
        [  3613, 54186,  2395],
        [     9,    54,   705],
    ]),
    'Extra Trees': np.array([
        [162133, 17034,  1119],
        [ 14074, 40949,  5171],
        [     5,    36,   732],
    ]),
    'Logistic Regression': np.array([
        [164398, 15629,   259],
        [  5318, 52123,  2753],
        [    12,    46,   710],
    ]),
    'Decision Tree': np.array([
        [164563, 15493,   230],
        [  5054, 52997,  2143],
        [     8,    70,   690],
    ]),
}

classes = ['Normal (0)', 'High Vol (1)', 'Crash (3)']

# ============================================================
# COMPUTE METRICS
# ============================================================

records_mcr = []   # Misclassification Rate
records_fpr = []   # False Positive Rate
records_fnr = []   # False Negative Rate

for model, cm in cms.items():
    total = cm.sum()
    col_sums = cm.sum(axis=0)   # predicted per class
    row_sums = cm.sum(axis=1)   # actual per class

    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fn = row_sums[i] - tp                        # missed actual positives
        fp = col_sums[i] - tp                        # false alarms
        tn = total - tp - fn - fp

        mcr = (fn + fp) / total                      # misclassification rate
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

        records_mcr.append({'Model': model, 'Class': cls,
                             'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
                             'MCR': mcr})
        records_fpr.append({'Model': model, 'Class': cls, 'FPR': fpr})
        records_fnr.append({'Model': model, 'Class': cls, 'FNR': fnr})

df_mcr = pd.DataFrame(records_mcr)
df_fpr = pd.DataFrame(records_fpr)
df_fnr = pd.DataFrame(records_fnr)

# ============================================================
# PIVOT FOR DISPLAY
# ============================================================

mcr_table = df_mcr.pivot(index='Model', columns='Class', values='MCR')[classes] * 100
fpr_table = df_fpr.pivot(index='Model', columns='Class', values='FPR')[classes] * 100
fnr_table = df_fnr.pivot(index='Model', columns='Class', values='FNR')[classes] * 100

print("=" * 65)
print("  PER-CLASS MISCLASSIFICATION RATE (%) — lower is better")
print("=" * 65)
print(mcr_table.round(4).to_string())

print("\n" + "=" * 65)
print("  PER-CLASS FALSE POSITIVE RATE (%) — lower is better")
print("=" * 65)
print(fpr_table.round(4).to_string())

print("\n" + "=" * 65)
print("  PER-CLASS FALSE NEGATIVE RATE (%) — lower is better")
print("=" * 65)
print(fnr_table.round(4).to_string())

  PER-CLASS MISCLASSIFICATION RATE (%) — lower is better
Class                Normal (0)  High Vol (1)  Crash (3)
Model                                                   
Decision Tree            8.6156        9.4343     1.0160
Extra Trees             13.3602       15.0527     2.6242
Logistic Regression      8.7951        9.8430     1.2725
Random Forest           10.2674       11.1781     1.1196
XGBoost                  8.8826        9.7382     1.0927

  PER-CLASS FALSE POSITIVE RATE (%) — lower is better
Class                Normal (0)  High Vol (1)  Crash (3)
Model                                                   
Decision Tree            8.3035        8.5958     0.9868
Extra Trees             23.0928        9.4279     2.6156
Logistic Regression      8.7432        8.6576     1.2525
Random Forest            5.9414       11.5761     1.0970
XGBoost                  7.5032        9.1841     1.0716

  PER-CLASS FALSE NEGATIVE RATE (%) — lower is better
Class                Normal (0)  Hi

In [2]:
# ============================================================
# COMBINED SUMMARY TABLE
# ============================================================

summary_rows = []
for model in cms:
    row = {'Model': model}
    for cls in classes:
        short = cls.split()[0][:4] + cls.split()[1][1] if '(' in cls else cls
        tag   = cls
        row[f'MCR {tag}'] = mcr_table.loc[model, tag]
        row[f'FPR {tag}'] = fpr_table.loc[model, tag]
        row[f'FNR {tag}'] = fnr_table.loc[model, tag]
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index('Model')

# Multi-level column display
arrays = [
    [c.split(' (')[0] for c in summary.columns],
    [c.split(' ')[0]  for c in summary.columns],
]
summary.columns = pd.MultiIndex.from_arrays(arrays)
print("\n" + "=" * 90)
print("  FULL SUMMARY TABLE (all rates in %)")
print("=" * 90)
print(summary.round(4).to_string())


  FULL SUMMARY TABLE (all rates in %)
                    MCR Normal FPR Normal FNR Normal MCR High Vol FPR High Vol FNR High Vol MCR Crash FPR Crash FNR Crash
                           MCR        FPR        FNR          MCR          FPR          FNR       MCR       FPR       FNR
Model                                                                                                                    
XGBoost                 8.8826     7.5032     9.3490       9.7382       9.1841      11.4048    1.0927    1.0716    7.6923
Random Forest          10.2674     5.9414    11.7303      11.1781      11.5761       9.9811    1.1196    1.0970    8.2031
Extra Trees            13.3602    23.0928    10.0690      15.0527       9.4279      31.9716    2.6242    2.6156    5.3040
Logistic Regression     8.7951     8.7432     8.8127       9.8430       8.6576      13.4083    1.2725    1.2525    7.5521
Decision Tree           8.6156     8.3035     8.7211       9.4343       8.5958      11.9563    1.0160    0.